# HW2 · 01 Data cleaning and validation
**Yixiao Yang · UCB MFE Python Pre-Program, Fall 2026**

This notebook reads the supplied `ohlc_hw` delivery, preserves its diagnosis, and writes a validated `ohlc_hw_cleaned` table to a **local working copy**. Run from this homework directory. The input is the actual bCourses database supplied for this assignment; there is no automatic synthetic fallback.

**Data provenance:** course `data.db`, uploaded 2026-10-28. The original file is opened read-only; cleaning writes only to the homework working copy. The separate injected examples are explicitly synthetic and illustrate edge cases.

All cleaning and demonstration functions are defined in this notebook. No adjacent Python helper module is required. The first run creates `data/data.db` from `source_db`; later runs reuse that working copy.

In [1]:
source_db = "../../../Lectures/Lecture 4/data/data.db"
db_path = "data/data.db"
audit_dir = "outputs/cleaning"
dataset_label = "COURSE DATABASE"

In [2]:
# Parameters
db_path = "data/data.db"
audit_dir = "outputs/cleaning"


### Cleaning functions: input schema and normalization

In [3]:
"""HW2 cleaning: diagnose first, repair conservatively, retain a row-level audit."""

from contextlib import closing
from dataclasses import dataclass
from html import unescape
from pathlib import Path
import json
import re
import sqlite3

import numpy as np
import pandas as pd

RAW = ['open', 'high', 'low', 'close']
PRICES = RAW + ['adj_close']
KEY = ['ticker', 'ts']
CORE = ['ts', 'ticker'] + PRICES + ['volume']
DEDUP = KEY + ['payload_signature']
ISSUES = ['ticker', 'issue_type', 'issue_date', 'description', 'severity']

def normalize_ticker(value):
    """Remove vendor HTML/entities and whitespace, preserving ticker punctuation."""
    if pd.isna(value):
        return pd.NA
    value = re.sub(r'<[^>]*>', '', unescape(str(value)))
    value = re.sub(r'\s+', '', value).upper()
    return value if value else pd.NA


def normalize_frame(frame):
    """Parse the supplied schema; retain original labels/dates and stable row IDs."""
    missing = set(CORE) - set(frame.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    out = frame[CORE].copy().reset_index(drop=True)
    out.insert(0, 'source_row_id', np.arange(len(out), dtype=int))
    out['ticker_original'] = out['ticker']
    out['ts_original'] = out['ts'].astype('string')
    # Preserve payload identity before numeric parsing; distinct bad strings must
    # not collapse into an alleged exact duplicate merely because both become NaN.
    out['payload_signature'] = pd.util.hash_pandas_object(out[PRICES + ['volume']], index=False).astype('string')
    out['ticker'] = out['ticker'].map(normalize_ticker).astype('string')
    out['ts'] = pd.to_datetime(out['ts'], errors='coerce', format='mixed')
    # These are daily observations. Intraday/time-zone inputs are not silently rounded.
    if not isinstance(out['ts'].dtype, pd.DatetimeTZDtype):
        out.loc[out['ts'].ne(out['ts'].dt.normalize()), 'ts'] = pd.NaT
    else:
        raise ValueError('Timezone-aware timestamps are outside this daily schema.')
    for column in PRICES + ['volume']:
        out[column] = pd.to_numeric(out[column], errors='coerce').astype(float)
    return out


def reference_rows(reference):
    """Return unique trusted keys; never choose arbitrarily among conflicting rows."""
    ref = normalize_frame(reference).dropna(subset=KEY).drop_duplicates(DEDUP)
    conflicts = ref.duplicated(KEY, keep=False)
    return ref.loc[~conflicts].sort_values(KEY).copy()

### Cleaning functions: sessions, return screening and quality diagnostics

In [4]:
def missing_sessions(frame, reference):
    """Compare each ticker with its full supplied reference history, not weekdays."""
    expected = normalize_frame(reference)[KEY].dropna().drop_duplicates()
    observed = frame[KEY].dropna().drop_duplicates()
    gaps = expected.merge(observed.assign(present=True), on=KEY, how='left')
    gaps = gaps.loc[gaps['present'].isna(), KEY].sort_values(KEY)
    counts = gaps.groupby('ticker').size()
    summary = expected.groupby('ticker')['ts'].agg(start='min', end='max')
    summary['missing_trading_days'] = counts.reindex(summary.index, fill_value=0)
    summary['description'] = [
        f'Ticker {t} is missing {r.missing_trading_days} trading days between '
        f'{r.start:%Y-%m-%d} and {r.end:%Y-%m-%d}'
        for t, r in summary.iterrows()
    ]
    return gaps.reset_index(drop=True), summary.reset_index()


def simple_return_screen(frame):
    """Two-sided within-ticker 3-sigma screen, sample SD; undefined SD => no flag.

    Exact duplicates are removed and conflicting keys excluded before returns.
    A missing/invalid close breaks the return chain: no implicit price filling.
    """
    out = frame.dropna(subset=KEY).drop_duplicates(DEDUP).copy()
    out = out.loc[~out.duplicated(KEY, keep=False)].sort_values(KEY)
    close = out['close'].where(np.isfinite(out['close']) & out['close'].gt(0))
    ret = close.groupby(out['ticker']).pct_change(fill_method=None)
    ret = ret.where(np.isfinite(ret))
    groups = ret.groupby(out['ticker'])
    mean = groups.transform('mean')
    std = groups.transform(lambda s: s.std(ddof=1))
    out['simple_return'] = ret
    out['return_mean'] = mean
    out['return_std'] = std
    out['extreme_return'] = ret.notna() & std.gt(0) & (ret - mean).abs().gt(3 * std)
    return out


def diagnose(frame, reference):
    """Create the required long-form issue table; several issues may share a row."""
    data = frame.copy()
    parts = []

    def add(mask, issue, description, severity='high'):
        rows = data.loc[mask.fillna(False), KEY].rename(columns={'ts': 'issue_date'})
        rows = rows.assign(issue_type=issue, description=description, severity=severity)
        parts.append(rows[ISSUES])

    for column in CORE:
        add(data[column].isna(), 'missing_' + column, f'{column} is missing or unparseable')
    for column in PRICES:
        add(data[column].notna() & (~np.isfinite(data[column]) | data[column].le(0)),
            'invalid_' + column, f'{column} is nonpositive or non-finite')
    trusted = reference_rows(reference)[KEY + ['close', 'adj_close']].rename(
        columns={'close': 'reference_close', 'adj_close': 'reference_adj_close'})
    matched = data[KEY + ['close', 'adj_close']].merge(trusted, on=KEY, how='left', validate='many_to_one')
    trusted_valid = (np.isfinite(matched[['reference_close', 'reference_adj_close']]).all(axis=1)
                     & matched['reference_close'].gt(0) & matched['reference_adj_close'].gt(0))
    expected_adj = matched['close'] * matched['reference_adj_close'] / matched['reference_close']
    inconsistent = (trusted_valid & np.isfinite(matched['close']) & matched['close'].gt(0)
                    & np.isfinite(matched['adj_close']) & matched['adj_close'].gt(0)
                    & ~np.isclose(matched['adj_close'], expected_adj, rtol=1e-8, atol=1e-10))
    add(pd.Series(inconsistent.to_numpy(), index=data.index), 'adjustment_inconsistency',
        'Adjusted close differs from raw close times the trusted same-date factor; inspect both fields')
    add(data['high'].lt(data['close']), 'high_below_close', 'high < close')
    add(data['low'].gt(data['close']), 'low_above_close', 'low > close')
    add(data['high'].lt(data['open']), 'high_below_open', 'high < open')
    add(data['low'].gt(data['open']), 'low_above_open', 'low > open')
    add(data['high'].lt(data['low']), 'high_below_low', 'high < low')
    add(data['volume'].notna() & (~np.isfinite(data['volume']) | data['volume'].lt(0)),
        'invalid_volume', 'volume is negative or non-finite')
    unknown = data.get('volume_unknown', pd.Series(False, index=data.index)).astype(bool)
    add(data['volume'].eq(0) & ~unknown, 'zero_volume',
        'Observed zero volume: review for stale trading; not automatically invalid', 'medium')
    add(unknown, 'volume_unknown', 'Zero is a repair placeholder, not observed no-trading', 'medium')
    add(data['ticker'].ne(data['ticker_original'].astype('string')),
        'ticker_normalization', 'Markup, whitespace or case normalized; original label retained', 'low')
    add(data.duplicated(DEDUP), 'exact_duplicate', 'Redundant canonical observation', 'low')
    unique = data.drop_duplicates(DEDUP)
    conflict_ids = unique.loc[unique.duplicated(KEY, keep=False), 'source_row_id']
    add(data['source_row_id'].isin(conflict_ids), 'conflicting_key',
        'Different payloads share a normalized ticker/date: quarantine all candidates')
    screened = simple_return_screen(data)
    extreme_ids = screened.loc[screened['extreme_return'], 'source_row_id']
    add(data['source_row_id'].isin(extreme_ids), 'extreme_return',
        'abs(simple raw-close return - ticker mean) > 3 * sample SD; review only', 'medium')
    gaps, _ = missing_sessions(data, reference)
    parts.append(gaps.rename(columns={'ts': 'issue_date'}).assign(
        issue_type='missing_session', description='Absent relative to supplied ticker history',
        severity='medium')[ISSUES])
    # A ticker without a reference calendar is unassessed, not reported as zero gaps.
    ref_names = set(normalize_frame(reference)['ticker'].dropna())
    add(~data['ticker'].isin(ref_names), 'calendar_unavailable',
        'No supplied calendar for this ticker; session completeness cannot be assessed', 'medium')
    return pd.concat(parts, ignore_index=True)[ISSUES]

### Cleaning functions: conservative repairs and row accounting

In [5]:
def interpolate_confirmed_errors(prices, confirmed_errors):
    """Interpolate flagged confirmed errors only, between finite positive neighbours.

    Caller supplies evidence for the Boolean mask. Time interpolation is retrospective;
    leading/trailing errors stay missing. Unflagged missing/invalid cells are not repaired.
    """
    if not isinstance(prices.index, pd.DatetimeIndex) or not prices.index.is_monotonic_increasing:
        raise ValueError('Expected one ticker with a sorted DatetimeIndex.')
    if prices.index.has_duplicates:
        raise ValueError('Duplicate timestamps cannot be interpolated.')
    mask = confirmed_errors.reindex(prices.index, fill_value=False).astype(bool)
    valid = prices.where(np.isfinite(prices) & prices.gt(0) & ~mask)
    candidate = valid.interpolate(method='time', limit_area='inside')
    return prices.mask(mask, candidate)


def valid_raw(values):
    """Check finite positive O/H/L/C and both open/close bounds."""
    o, h, l, c = values
    return bool(np.isfinite(values).all() and min(values) > 0 and l <= o <= h and l <= c <= h)


@dataclass
class CleaningResult:
    """Clean data plus complete audit products; raw input is never modified."""
    cleaned: pd.DataFrame
    audit: pd.DataFrame
    before: pd.DataFrame
    after: pd.DataFrame
    gap_summary: pd.DataFrame
    outlier_review: pd.DataFrame
    counts: dict


def clean_data(raw, reference):
    """Normalize, diagnose, remove exact copies, and conservatively repair prices.

    All raw-price-missing rows are removed before filling. Unresolved or conflicting
    rows are quarantined. The reference contributes dates, date-specific adjustment
    factors and corroborating evidence, never a wholesale replacement dataset.
    """
    data = normalize_frame(raw)
    before = diagnose(data, reference)
    _, gap_summary = missing_sessions(data, reference)
    ref = reference_rows(reference)
    ref_map = {(row['ticker'], row['ts']): row for row in ref[KEY + PRICES].to_dict('records')}
    audit = data.copy().set_index('source_row_id', drop=False)
    audit['disposition'] = 'pending'
    audit['reason'] = ''
    audit['repair_flags'] = '[]'
    dispositions = np.full(len(raw), 'pending', dtype=object)
    reasons = np.full(len(raw), '', dtype=object)
    all_flags = np.full(len(raw), '[]', dtype=object)

    def dispose(ids, status, reason):
        indexes = np.asarray(ids, dtype=int)
        dispositions[indexes] = status
        reasons[indexes] = reason

    invalid_key = data[KEY].isna().any(axis=1)
    dispose(data.loc[invalid_key, 'source_row_id'], 'quarantined', 'invalid_key')
    work = data.loc[~invalid_key].copy()
    duplicates = work.duplicated(DEDUP)
    dispose(work.loc[duplicates, 'source_row_id'], 'removed', 'exact_duplicate')
    work = work.loc[~duplicates]
    conflicts = work.duplicated(KEY, keep=False)
    dispose(work.loc[conflicts, 'source_row_id'], 'quarantined', 'conflicting_key')
    work = work.loc[~conflicts]
    all_missing = work[RAW].isna().all(axis=1)
    dispose(work.loc[all_missing, 'source_row_id'], 'removed', 'all_raw_prices_missing')
    work = work.loc[~all_missing].sort_values(KEY)
    # Screen the original deduplicated timeline before removing all-missing rows.
    screened = simple_return_screen(data)
    extreme_ids = set(screened.loc[screened['extreme_return'], 'source_row_id'])
    kept, review = [], []
    for symbol, group in work.groupby('ticker', sort=True):
        previous_valid = None
        for row in group.to_dict('records'):
            rid, date = row['source_row_id'], row['ts']
            flags = []
            trusted = ref_map.get((symbol, date))
            original = np.array([row[c] for c in RAW], dtype=float)
            values = original.copy()
            is_extreme = rid in extreme_ids
            decision = 'not_flagged'
            if is_extreme:
                matched = (trusted is not None and np.isfinite(trusted['close'])
                           and trusted['close'] > 0)
                agrees = matched and np.isclose(row['close'], trusted['close'], rtol=1e-8, atol=1e-10)
                decision = ('reference_close_agrees; event_type_unverified' if agrees else
                            'reference_close_disagrees; quarantine' if matched else
                            'no_reference_evidence; retain_with_uncertainty_if_valid')
                review.append({'ticker': symbol, 'ts': date, 'source_row_id': rid,
                               'decision': decision, 'raw_close': row['close'],
                               'reference_close': trusted['close'] if matched else np.nan})
                if matched and not agrees:
                    dispose([rid], 'quarantined', 'extreme_return_reference_disagreement')
                    continue
            # Invalid scalar prices become missing; only earlier retained rows can donate.
            for j, column in enumerate(RAW):
                if not np.isfinite(values[j]) or values[j] <= 0:
                    flags.append(column + '_ffill')
                    values[j] = np.nan if previous_valid is None else previous_valid[j]
            if not np.isfinite(values).all():
                dispose([rid], 'quarantined', 'unresolved_leading_price_gap')
                continue
            if not valid_raw(values):
                # Option 3 is defensible only if O/H/L are independently corroborated.
                o, h, l, c = values
                ohl_agrees = (trusted is not None and valid_raw(np.array([trusted[c] for c in RAW]))
                              and np.allclose(values[:3], [trusted[c] for c in RAW[:3]],
                                              rtol=1e-8, atol=1e-10))
                close_was_present = np.isfinite(original[3]) and original[3] > 0
                if ohl_agrees and close_was_present and l <= o <= h and not l <= c <= h:
                    values[3] = (h + l) / 2
                    flags.append('close_ohl_midpoint_reference_corroborated')
                else:
                    dispose([rid], 'quarantined', 'unresolved_ohlc_bounds_after_fill')
                    continue
            if not valid_raw(values):
                raise AssertionError('A price repair escaped the OHLC check.')
            close_changed = not np.isclose(values[3], original[3], rtol=1e-10, atol=1e-12, equal_nan=True)
            factor = np.nan
            factor_source = 'unavailable'
            if trusted is not None:
                rc, ra = trusted['close'], trusted['adj_close']
                if np.isfinite([rc, ra]).all() and min(rc, ra) > 0:
                    factor, factor_source = ra / rc, 'ohlc_same_ticker_same_date'
            # Valid original vendor pair is retained only when close has not changed.
            if not np.isfinite(factor) and not close_changed and np.isfinite(row['adj_close']) and row['adj_close'] > 0:
                factor = row['adj_close'] / values[3]
                factor_source = 'original_valid_vendor_pair_unverified'
            adjusted = values[3] * factor
            if not np.isfinite(adjusted) or adjusted <= 0:
                dispose([rid], 'quarantined', 'unresolved_adjustment_factor')
                continue
            adj_changed = close_changed or not np.isclose(row['adj_close'], adjusted, rtol=1e-8, atol=1e-10)
            if adj_changed:
                flags.append('adj_close_reconstructed')
            else:
                # Avoid unflagged floating-point perturbations of a valid vendor value.
                adjusted = row['adj_close']
            volume_unknown = not np.isfinite(row['volume']) or row['volume'] < 0
            if volume_unknown:
                row['volume'] = 0.0
                flags.append('volume_unknown_zero_placeholder')
            for j, column in enumerate(RAW):
                row['original_' + column] = original[j]
                row[column] = values[j]
            row['original_adj_close'] = row['adj_close']
            row.update(adj_close=adjusted, adjustment_factor=factor, factor_source=factor_source,
                       volume_unknown=volume_unknown, raw_price_repaired=bool(not np.array_equal(values, original)),
                       close_repaired=close_changed, adj_close_repaired=adj_changed,
                       extreme_return_before=is_extreme, outlier_decision=decision,
                       repair_flags=json.dumps(flags))
            previous_valid = values.copy()
            kept.append(row)
            dispose([rid], 'retained', 'validated')
            all_flags[rid] = json.dumps(flags)
    audit['disposition'], audit['reason'], audit['repair_flags'] = dispositions, reasons, all_flags
    cleaned = pd.DataFrame(kept)
    if cleaned.empty:
        cleaned = work.iloc[:0].copy()
        for column in ['adjustment_factor', 'factor_source', 'volume_unknown', 'raw_price_repaired',
                       'close_repaired', 'adj_close_repaired', 'extreme_return_before',
                       'outlier_decision', 'repair_flags']:
            cleaned[column] = pd.Series(dtype='object')
    else:
        cleaned = cleaned.sort_values(KEY).reset_index(drop=True)
    price_changed = cleaned['close_repaired'].astype(bool) | cleaned['adj_close_repaired'].astype(bool)
    cleaned['return_affected_by_repair'] = price_changed | price_changed.groupby(
        cleaned['ticker']).shift(1, fill_value=False).astype(bool)
    post_screen = simple_return_screen(cleaned)
    flagged = set(post_screen.loc[post_screen['extreme_return'], 'source_row_id'])
    cleaned['extreme_return_after'] = cleaned['source_row_id'].isin(flagged)
    validate_cleaned(cleaned)
    after = diagnose(cleaned, reference)
    counts = {'input': len(raw), **audit['disposition'].value_counts().to_dict()}
    for name in ['retained', 'removed', 'quarantined']:
        counts.setdefault(name, 0)
    assert counts['input'] == counts['retained'] + counts['removed'] + counts['quarantined']
    assert not audit['disposition'].eq('pending').any()
    review = pd.DataFrame(review, columns=['ticker', 'ts', 'source_row_id', 'decision', 'raw_close', 'reference_close'])
    return CleaningResult(cleaned, audit.reset_index(drop=True), before, after, gap_summary, review, counts)

### Cleaning functions: validation and local database persistence

In [6]:
def validate_cleaned(frame):
    """Fail explicitly on invalid retained prices, volume, keys, dates or adjustment."""
    if frame[KEY].isna().any().any() or frame.duplicated(KEY).any():
        raise ValueError('Cleaned keys must be nonmissing and unique.')
    if frame.empty:
        return
    prices = frame[PRICES].to_numpy(float)
    if not np.isfinite(prices).all() or not (prices > 0).all():
        raise ValueError('Cleaned prices must be finite and positive.')
    if not frame['open'].between(frame['low'], frame['high']).all() or not frame['close'].between(frame['low'], frame['high']).all():
        raise ValueError('Cleaned OHLC bounds failed.')
    if not np.isfinite(frame['volume']).all() or frame['volume'].lt(0).any():
        raise ValueError('Volume must be finite and nonnegative.')
    if not np.allclose(frame['adj_close'], frame['close'] * frame['adjustment_factor'], rtol=1e-8):
        raise ValueError('Adjusted close is inconsistent with the recorded date-specific factor.')


def open_database(path, readonly=True):
    """Open an existing DB without accidentally creating an empty SQLite file."""
    path = Path(path).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f'Missing {path}. Supply the bCourses data.db; no synthetic fallback.')
    return sqlite3.connect(path.as_uri() + ('?mode=ro' if readonly else '?mode=rw'), uri=True)


def load_tables(path):
    """Read the homework delivery and trusted reference from an existing local copy."""
    with closing(open_database(path)) as conn:
        raw = pd.read_sql_query('SELECT * FROM ohlc_hw', conn)
        reference = pd.read_sql_query('SELECT * FROM ohlc', conn)
    return raw, reference


def save_cleaned(result, db_path, audit_dir):
    """Persist cleaned/audit tables, ISO dates and CSV diagnostics; preserve raw table."""
    validate_cleaned(result.cleaned)
    folder = Path(audit_dir)
    folder.mkdir(parents=True, exist_ok=True)
    tables = {'ohlc_hw_cleaned': result.cleaned, 'ohlc_hw_cleaning_audit': result.audit,
              'ohlc_hw_quality_before': result.before, 'ohlc_hw_quality_after': result.after}
    with closing(open_database(db_path, readonly=False)) as conn:
        for name, frame in tables.items():
            stored = frame.copy()
            for c in ['ts', 'issue_date']:
                if c in stored:
                    stored[c] = pd.to_datetime(stored[c]).dt.strftime('%Y-%m-%d')
            stored.to_sql(name, conn, if_exists='replace', index=False)
        conn.execute('CREATE UNIQUE INDEX IF NOT EXISTS hw2_cleaned_key ON ohlc_hw_cleaned(ticker, ts)')
        conn.commit()
        roundtrip = pd.read_sql_query('SELECT * FROM ohlc_hw_cleaned ORDER BY ticker, ts', conn)
        validate_cleaned(roundtrip)
        if len(roundtrip) and not roundtrip['ts'].str.fullmatch(r'\d{4}-\d{2}-\d{2}').all():
            raise AssertionError('SQL dates must be ISO dates.')
    for name, frame in [('quality_before', result.before), ('quality_after', result.after),
                        ('row_audit', result.audit), ('missing_sessions', result.gap_summary),
                        ('outlier_review', result.outlier_review)]:
        frame.to_csv(folder / f'{name}.csv', index=False)
    (folder / 'row_accounting.json').write_text(json.dumps(result.counts, indent=2), encoding='utf-8')


def prepare_working_database(source_db, db_path):
    """Create a local SQLite backup if absent; never clean the source database."""
    target = Path(db_path).expanduser().resolve()
    source = Path(source_db).expanduser().resolve() if source_db else None
    if source is not None and (source == target or (
            source.exists() and target.exists() and source.samefile(target))):
        raise ValueError('source_db and db_path must refer to different files.')
    if target.exists():
        with closing(open_database(target)):
            pass
        return target
    if source is None:
        raise FileNotFoundError('Provide source_db to create the missing working copy.')
    with closing(open_database(source)) as src:
        target.parent.mkdir(parents=True, exist_ok=True)
        with closing(sqlite3.connect(target)) as dst:
            src.backup(dst)
    return target

### Small synthetic demonstrations (separate from the course data)

In [7]:
def tiny_frame(ticker='DEMO_A', periods=6):
    """Build a known-good tiny synthetic history with an explicit session calendar."""
    dates = pd.date_range('2024-01-02', periods=periods, freq='B')
    return pd.DataFrame({'ts': dates, 'ticker': ticker, 'open': 100., 'high': 105.,
                         'low': 95., 'close': 100., 'adj_close': 50., 'volume': 1000.})


def injected_examples():
    """Exercise grouped fill, all-missing removal, bounds, duplicate conflicts and gaps."""
    cases = []
    def check(name, raw, ref, expected_retained, expected_removed=0, expected_quarantined=0):
        result = clean_data(raw, ref)
        assert result.counts['retained'] == expected_retained, (name, result.counts)
        assert result.counts['removed'] == expected_removed, (name, result.counts)
        assert result.counts['quarantined'] == expected_quarantined, (name, result.counts)
        cases.append({'case': name, **result.counts, 'status': 'PASS'})
        return result

    ref = tiny_frame()
    raw = ref.copy()
    raw.loc[1, ['open', 'close', 'adj_close', 'volume']] = np.nan
    result = check('Missing cells: prior same-ticker price and date-specific factor', raw, ref, 6)
    assert result.cleaned.loc[1, 'volume_unknown']
    assert result.cleaned.loc[1, 'adj_close'] == 50
    raw = ref.copy()
    raw.loc[2, 'adj_close'] = 700.
    result = check('Positive adjusted-price corruption: reconcile trusted date factor', raw, ref, 6)
    assert result.before.issue_type.eq('adjustment_inconsistency').sum() == 1
    assert not result.after.issue_type.eq('adjustment_inconsistency').any()
    assert result.cleaned.loc[2, 'adj_close'] == 50.
    np.testing.assert_array_equal(result.cleaned[['open','high','low','close']], ref[['open','high','low','close']])
    raw = ref.copy()
    raw.loc[1, ['open', 'high', 'low', 'close']] = np.nan
    check('All OHLC missing: remove before fill', raw, ref, 5, 1)
    other = tiny_frame('DEMO_B')
    raw = pd.concat([ref, other], ignore_index=True)
    raw.loc[6, 'open'] = np.nan
    check('Leading gap: never borrow another ticker', raw, pd.concat([ref, other]), 11, 0, 1)
    raw = ref.copy()
    raw.loc[1, ['open', 'high', 'low', 'close']] = [120, 125, 115, np.nan]
    check('Forward-fill outside current range: quarantine', raw, ref, 5, 0, 1)
    raw = ref.copy()
    raw.loc[2, 'close'] = 200.
    check('Impossible close with corroborated OHL: midpoint repair', raw, ref, 6)
    raw = ref.copy()
    raw.loc[1, ['open', 'high', 'low', 'close', 'adj_close', 'volume']] = [0, np.inf, -1, 100, -2, np.inf]
    result = check('Nonpositive/nonfinite prices and volume', raw, ref, 6)
    assert result.cleaned.loc[1, 'volume_unknown']
    raw = pd.concat([ref, ref.iloc[[1]].assign(ticker=' <b>demo_a</b> ')], ignore_index=True)
    check('Normalize labels before exact duplicate removal', raw, ref, 6, 1)
    raw = pd.concat([ref, ref.iloc[[1]].assign(volume=2000)], ignore_index=True)
    check('Conflicting keys: quarantine both candidates', raw, ref, 5, 0, 2)
    raw = ref.drop(index=2)
    gaps, summary = missing_sessions(normalize_frame(raw), ref)
    assert gaps.ts.tolist() == [ref.loc[2, 'ts']]
    assert summary.missing_trading_days.iloc[0] == 1
    check('One missing session: report without inventing prices', raw, ref, 5)
    split = ref.copy()
    split.loc[:2, ['open', 'high', 'low', 'close']] *= 2
    split['adj_close'] = 100.
    raw = split.copy()
    raw.loc[3, 'adj_close'] = np.nan
    result = check('Split-date factor: no factor forward-fill', raw, split, 6)
    assert result.cleaned.loc[3, 'adj_close'] == 100.
    raw = ref.copy()
    raw.loc[0, 'adj_close'] = np.nan
    missing_factor = ref.copy()
    missing_factor.loc[0, 'adj_close'] = np.nan
    check('Unresolved adjusted-price convention: quarantine', raw, missing_factor, 5, 0, 1)
    # A statistical outlier agrees with the reference: retain without inventing news.
    long = tiny_frame(periods=101)
    close = 100 * np.cumprod(np.r_[1., np.repeat(1.001, 49), 1.5, np.repeat(1.001, 49), .5])
    long['close'] = close
    long['open'] = close
    long['high'] = close * 1.01
    long['low'] = close * .99
    long['adj_close'] = close
    screen = simple_return_screen(normalize_frame(long))
    assert screen.loc[screen.extreme_return, 'simple_return'].gt(0).any()
    assert screen.loc[screen.extreme_return, 'simple_return'].lt(0).any()
    check('Two-sided 3-sigma flags are retained when source corroborates', long, long, len(long))
    flat = simple_return_screen(normalize_frame(ref))
    assert not flat.extreme_return.any()
    prices = pd.Series([900., 100., 999., 104., 800.], index=pd.date_range('2024-01-01', periods=5))
    evidence_mask = pd.Series([True, False, True, False, True], index=prices.index)
    repaired = interpolate_confirmed_errors(prices, evidence_mask)
    assert repaired.iloc[2] == 102 and pd.isna(repaired.iloc[0]) and pd.isna(repaired.iloc[-1])
    cases.append({'case': 'Confirmed-error interpolation: inside only; no extrapolation', 'status': 'PASS'})
    return pd.DataFrame(cases)

In [8]:
from pathlib import Path
import sqlite3
from contextlib import closing
import numpy as np
import pandas as pd
from IPython.display import display
pd.set_option('display.max_columns', 18)
print("DATASET:", dataset_label)
working_database = prepare_working_database(source_db, db_path)
print("Working copy:", Path(db_path))
raw, reference = load_tables(working_database)

DATASET: COURSE DATABASE
Working copy: data/data.db


## 1. Load and inspect
| Column | Meaning |
|---|---|
| `ts` | Daily observation date; stored as ISO date after cleaning |
| `ticker` | Vendor symbol; normalize HTML, whitespace and case before grouping |
| `open`, `high`, `low`, `close` | Raw daily equity prices |
| `adj_close` | Vendor adjusted close; raw OHLC bounds do not apply |
| `volume` | Daily share volume; missing volume is unknown, not no trading |

The source is kept intact. `source_row_id` identifies its row in this load; original labels and dates travel with the cleaned data and the audit.

In [9]:
raw.info()
display(raw.describe(include='all'))
display(raw.head())
print("Missing cells in original delivery:")
display(raw.isna().sum().rename('missing_cells').to_frame())
normalized = normalize_frame(raw)
print("Parsed date range:", normalized.ts.min(), "to", normalized.ts.max())
print("Raw labels:", raw.ticker.nunique(), "| Normalized tickers:", normalized.ticker.nunique())
print("Tickers:", sorted(normalized.ticker.dropna().unique()))
display(normalized[['ticker_original','ticker']].drop_duplicates().sort_values('ticker'))
display(normalized.groupby('ticker').agg(first_date=('ts','min'), last_date=('ts','max'), rows=('ts','size')))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  object 
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  object 
dtypes: float64(5), int64(1), object(2)
memory usage: 24.3+ MB
Missing cells in original delivery:
Parsed date range: 1962-01-02 00:00:00 to 2024-12-06 00:00:00
Raw labels: 150 | Normalized tickers: 30
Tickers: ['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SHW', 'TRV', 'UNH', 'V', 'VZ', 'WMT']


,ts,open,high,low,close,adj_close,volume,ticker
count,397711,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05,397711
unique,15868,NaN,NaN,NaN,NaN,NaN,NaN,150
top,2020-11-24,NaN,NaN,NaN,NaN,NaN,NaN,PG
freq,40,NaN,NaN,NaN,NaN,NaN,NaN,15450
mean,NaN,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07,NaN
std,NaN,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08,NaN
min,NaN,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00,NaN
25%,NaN,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06,NaN
50%,NaN,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06,NaN
75%,NaN,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07,NaN


,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


,missing_cells
ts,0
open,0
high,0
low,0
close,0
adj_close,19855
volume,0
ticker,0


,ticker_original,ticker
102,"<a target=""_blank""><span style=""color: blue"">A...",AAPL
4,AAPL,AAPL
744,"<a target=""_blank""><span style=""font-weight: b...",AAPL
1121,"<a target=""_blank""><span style=""color: red"">AA...",AAPL
825,"<a target=""_blank""><span style=""font-weight: b...",AAPL
...,...,...
2166,"<a target=""_blank""><span style=""font-weight: b...",WMT
1036,"<a target=""_blank""><span style=""color: blue"">W...",WMT
234,"<a target=""_blank""><span style=""color: red"">WM...",WMT
783,"<a target=""_blank""><span style=""font-weight: b...",WMT


,first_date,last_date,rows
ticker,,,
AAPL,1980-12-12,2024-10-18,11942
AMGN,1983-06-17,2024-10-18,11290
AMZN,1997-05-15,2024-12-06,7458
AXP,1972-04-03,2024-10-18,14260
BA,1962-01-02,2024-10-18,16980
CAT,1962-01-02,2024-10-18,17044
CRM,2004-06-23,2024-10-18,5525
CSCO,1990-02-16,2024-10-18,9427
CVX,1962-01-02,2024-12-06,17073


## 2. Diagnose before repairs
`diagnose` builds a long-form Data Quality Report: a row may have several issues. All diagnostic and repair functions are defined in the labeled cells above; rerunning this notebook requires no local helper modules.

- **High:** invalid or missing prices/keys, contradictory OHLC, conflicting duplicate keys and invalid volume require repair or quarantine.
- **Medium:** missing sessions, zero/unknown volume, missing reference calendars and 3σ flags require disclosure/review; statistical flags do not imply errors.
- **Low:** ticker formatting and removable exact duplicates have deterministic remedies.

Raw close simple returns are calculated after normalized exact deduplication and exclusion of conflicting keys, sorted within ticker. The screen is `abs(r - mean(r)) > 3 * std(r, ddof=1)` in both tails. Undefined returns are excluded; zero/undefined SD gives no flag. It is retrospective.

Positive adjusted prices can still be corrupted. A matched reference factor checks `adj_close` against raw close × same-date factor; discrepancies receive a separate high-severity `adjustment_inconsistency` issue. This does not apply raw OHLC bounds to adjusted close.

In [10]:
quality_before = diagnose(normalized, reference)
display(quality_before.head(25))
display(quality_before.groupby(['issue_type','severity']).size().rename('issue_count').to_frame())
assert quality_before.columns.tolist() == ['ticker','issue_type','issue_date','description','severity']
screen = simple_return_screen(normalized)
display(screen.loc[screen.extreme_return, ['ticker','ts','close','simple_return','return_mean','return_std']].head(25))

,ticker,issue_type,issue_date,description,severity
0,KO,missing_adj_close,2024-04-16,adj_close is missing or unparseable,high
1,NVDA,missing_adj_close,2018-06-12,adj_close is missing or unparseable,high
2,HD,missing_adj_close,2013-04-16,adj_close is missing or unparseable,high
3,PG,missing_adj_close,2015-03-20,adj_close is missing or unparseable,high
4,JPM,missing_adj_close,2018-03-13,adj_close is missing or unparseable,high
5,NVDA,missing_adj_close,2000-05-15,adj_close is missing or unparseable,high
6,CSCO,missing_adj_close,2001-10-02,adj_close is missing or unparseable,high
7,MMM,missing_adj_close,1975-07-03,adj_close is missing or unparseable,high
8,IBM,missing_adj_close,2019-06-21,adj_close is missing or unparseable,high
9,CSCO,missing_adj_close,2006-01-09,adj_close is missing or unparseable,high


,,issue_count
issue_type,severity,
adjustment_inconsistency,high,11265
exact_duplicate,low,29460
extreme_return,medium,5056
invalid_open,high,5
low_above_open,high,5
missing_adj_close,high,19855
ticker_normalization,low,39936
zero_volume,medium,81


,ticker,ts,close,simple_return,return_mean,return_std
393432,AAPL,1980-12-26,0.158480,0.092288,0.001074,0.027846
139446,AAPL,1981-04-02,0.117750,0.087659,0.001074,0.027846
107726,AAPL,1981-09-25,0.063616,-0.129776,0.001074,0.027846
176299,AAPL,1982-03-19,0.074219,0.090173,0.001074,0.027846
154954,AAPL,1982-11-22,0.125560,-0.089089,0.001074,0.027846
294878,AAPL,1982-11-30,0.142300,0.103871,0.001074,0.027846
86617,AAPL,1983-01-20,0.166850,0.111518,0.001074,0.027846
346813,AAPL,1983-04-20,0.226000,0.088684,0.001074,0.027846
382520,AAPL,1983-07-26,0.174670,-0.092718,0.001074,0.027846
34943,AAPL,1983-08-31,0.166290,0.133074,0.001074,0.027846


### Missing cells are different from absent sessions
The calendar is each ticker's complete supplied `ohlc` history. It is a **coverage assumption about this delivery**, not proof of exchange completeness. Weekdays alone would incorrectly count holidays. A zero result is legitimate. We list absent rows without creating prices, including reference dates entirely outside an observed ticker's shorter delivery.

In [11]:
gaps, gap_summary = missing_sessions(normalized, reference)
display(gap_summary)
for message in gap_summary.description:
    print(message)
display(gaps.head(20))

Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker GS is missing 0 trading days between 1999-05-04 and 2024-10-18
Ticker HD is missing 0 trading days between 1981-09-22 and 2024-10-18
Ticker HON is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker IBM is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticke

,ticker,start,end,missing_trading_days,description
0,AAPL,1980-12-12,2024-10-18,0,Ticker AAPL is missing 0 trading days between ...
1,AMGN,1983-06-17,2024-10-18,0,Ticker AMGN is missing 0 trading days between ...
2,AMZN,1997-05-15,2024-12-06,0,Ticker AMZN is missing 0 trading days between ...
3,AXP,1972-04-03,2024-10-18,0,Ticker AXP is missing 0 trading days between 1...
4,BA,1962-01-02,2024-10-18,0,Ticker BA is missing 0 trading days between 19...
5,CAT,1962-01-02,2024-10-18,0,Ticker CAT is missing 0 trading days between 1...
6,CRM,2004-06-23,2024-10-18,0,Ticker CRM is missing 0 trading days between 2...
7,CSCO,1990-02-16,2024-10-18,0,Ticker CSCO is missing 0 trading days between ...
8,CVX,1962-01-02,2024-12-06,0,Ticker CVX is missing 0 trading days between 1...
9,DIS,1962-01-02,2024-12-06,0,Ticker DIS is missing 0 trading days between 1...


,ticker,ts


## 3. Repair policy and evidence
1. Normalize labels and compare complete payloads. Remove exact duplicate observations; quarantine every candidate of an unresolved conflicting key.
2. Remove all-raw-OHLC-missing rows **before** filling. Fill missing/nonpositive/nonfinite raw prices using the preceding **retained, validated observation of that ticker**. Leading gaps remain unresolved. Check today's OHLC range after every candidate repair; quarantine violations.
3. For an impossible close, use the same-day H/L midpoint (assignment Option 3) **only** when trusted same-date `ohlc` corroborates O/H/L. Otherwise quarantine; a contradictory range does not tell us which field is wrong. Midpoints and fills are estimates, not recovered actual trades.
4. For a 3σ flag, retain a structurally valid price if the reference agrees, or retain with explicitly stated uncertainty if reference evidence is unavailable. A disagreement with a trusted reference is quarantined. Agreement does not establish a split, merger or news event. The injected demonstration shows bounded interpolation of independently confirmed errors; this pipeline does not interpolate unexplained statistical extremes.
5. Reconstruct adjusted close from validated close × **same-ticker, same-date** `ohlc.adj_close / ohlc.close`. Never forward-fill a factor across dates. If no factor exists, an original finite positive vendor pair may be retained only when raw close is unchanged, with an unverified-source label. Otherwise quarantine.
6. Replace missing/negative/nonfinite volume with zero and set `volume_unknown`; it is not evidence of no trading. Preserve repair flags and the original source observation in the audit.

No entire raw row is replaced by its reference row. Retained, removed and quarantined input records reconcile exactly.

In [12]:
result = clean_data(raw, reference)
cleaned = result.cleaned
display(pd.Series(result.counts, name='rows').to_frame())
assert result.counts['input'] == sum(result.counts[k] for k in ['retained','removed','quarantined'])
display(result.audit.groupby(['disposition','reason']).size().rename('rows').to_frame())
display(cleaned.loc[cleaned.repair_flags.ne('[]')].head(15))
print("Statistical outlier investigation (reference evidence; no invented event attribution):")
display(result.outlier_review.head(25))

Statistical outlier investigation (reference evidence; no invented event attribution):


,rows
input,397711
retained,368246
removed,29460
quarantined,5


,,rows
disposition,reason,
quarantined,unresolved_leading_price_gap,5
removed,exact_duplicate,29460
retained,validated,368246


,source_row_id,ts,ticker,open,high,low,close,adj_close,volume,...,volume_unknown,raw_price_repaired,close_repaired,adj_close_repaired,extreme_return_before,outlier_decision,repair_flags,return_affected_by_repair,extreme_return_after
17,241023,1981-01-08,AAPL,0.13505,0.13505,0.13505,0.13505,0.104100,39827236.0,...,False,False,False,True,False,not_flagged,"[""adj_close_reconstructed""]",True,False
20,229549,1981-01-13,AAPL,0.13616,0.13616,0.13616,0.13616,0.104960,23049620.0,...,False,False,False,True,False,not_flagged,"[""adj_close_reconstructed""]",True,False
35,129806,1981-02-03,AAPL,0.12333,0.12333,0.12333,0.12333,0.095069,19152016.0,...,False,False,False,True,False,not_flagged,"[""adj_close_reconstructed""]",True,False
42,36720,1981-02-12,AAPL,0.11663,0.11663,0.11663,0.11663,0.089907,14560012.0,...,False,False,False,True,False,not_flagged,"[""adj_close_reconstructed""]",True,False
86,58320,1981-04-16,AAPL,0.11161,0.11161,0.11161,0.11161,0.086035,23878420.0,...,False,False,False,True,False,not_flagged,"[""adj_close_reconstructed""]",True,False
93,16448,1981-04-28,AAPL,0.12612,0.12612,0.12612,0.12612,0.097220,32188832.0,...,False,False,False,True,False,not_flagged,"[""adj_close_reconstructed""]",True,False
101,277136,1981-05-08,AAPL,0.12500,0.12500,0.12500,0.12500,0.096360,7907204.0,...,False,False,False,True,False,not_flagged,"[""adj_close_reconstructed""]",True,False
108,173691,1981-05-19,AAPL,0.12277,0.12277,0.12277,0.12277,0.094639,25424024.0,...,False,False,False,True,False,not_flagged,"[""adj_close_reconstructed""]",True,False
119,373137,1981-06-04,AAPL,0.14341,0.14341,0.14341,0.14341,0.110560,56067255.0,...,False,False,False,True,False,not_flagged,"[""adj_close_reconstructed""]",True,False
146,236400,1981-07-14,AAPL,0.10603,0.10603,0.10603,0.10603,0.081734,19779216.0,...,False,False,False,True,False,not_flagged,"[""adj_close_reconstructed""]",True,False


,ticker,ts,source_row_id,decision,raw_close,reference_close
0,AAPL,1980-12-26,393432,reference_close_agrees; event_type_unverified,0.158480,0.158480
1,AAPL,1981-04-02,139446,reference_close_agrees; event_type_unverified,0.117750,0.117750
2,AAPL,1981-09-25,107726,reference_close_agrees; event_type_unverified,0.063616,0.063616
3,AAPL,1982-03-19,176299,reference_close_agrees; event_type_unverified,0.074219,0.074219
4,AAPL,1982-11-22,154954,reference_close_agrees; event_type_unverified,0.125560,0.125560
5,AAPL,1982-11-30,294878,reference_close_agrees; event_type_unverified,0.142300,0.142300
6,AAPL,1983-01-20,86617,reference_close_agrees; event_type_unverified,0.166850,0.166850
7,AAPL,1983-04-20,346813,reference_close_agrees; event_type_unverified,0.226000,0.226000
8,AAPL,1983-07-26,382520,reference_close_agrees; event_type_unverified,0.174670,0.174670
9,AAPL,1983-08-31,34943,reference_close_agrees; event_type_unverified,0.166290,0.166290


## 4. Validate again and compare
All retained raw/adjusted prices must be finite and positive; volume finite and nonnegative; open/close inside low/high; ticker/date keys unique. Adjusted close must agree with its saved factor. Medium review flags can legitimately remain. Quarantined/removed rows may create coverage gaps, which the post-cleaning report still discloses.

In [13]:
validate_cleaned(cleaned)
before_after = pd.concat({'before': normalized[PRICES + ['volume']].describe(),
                          'after': cleaned[PRICES + ['volume']].describe()}, axis=1)
display(before_after)
display(result.after.groupby(['issue_type','severity']).size().rename('remaining_issues').to_frame())
print("Remaining invalid retained price/volume cells:", int((~np.isfinite(cleaned[PRICES + ['volume']])).sum().sum()))
print("Unique keys:", not cleaned.duplicated(['ticker','ts']).any())
display(result.audit.loc[result.audit.disposition.eq('quarantined')].head(20))
print("Raw labels preserved:", 'ticker_original' in cleaned)

Remaining invalid retained price/volume cells: 0
Unique keys: True
Raw labels preserved: True


before                                               \
                open           high            low          close   
count  397711.000000  397711.000000  397711.000000  397711.000000   
mean       46.285479      46.747660      45.816279      46.292903   
std        67.132594      67.778210      66.477178      67.141879   
min         0.000000       0.005208       0.004801       0.005208   
25%         4.209938       4.240000       4.160000       4.200000   
50%        19.850000      20.060000      19.630000      19.860000   
75%        57.530000      58.130000      56.950000      57.530000   
max       604.260000     608.630000     597.880000     605.400000   

                                            after                 \
           adj_close        volume           open           high   
count  377856.000000  3.977110e+05  368246.000000  368246.000000   
mean       51.652250  3.097773e+07      46.309394      46.771824   
std       194.584045  1.266237e+08      67.143254      67.789319   
min         0.002945  0.000000e+00       0.005208       0.005208   
25%         1.390000  1.730700e+06       4.210000       4.250000   
50%        11.960000  4.648200e+06      19.890000      20.090000   
75%        43.770000  1.102380e+07      57.560000      58.160000   
max     10442.510783  9.276606e+09     604.260000     608.630000   

                                                                  
                 low          close      adj_close        volume  
count  368246.000000  368246.000000  368246.000000  3.682460e+05  
mean       45.840375      46.317325      37.274670  3.095184e+07  
std        66.489450      67.154939      63.587872  1.266153e+08  
min         0.004801       0.005208       0.002945  0.000000e+00  
25%         4.170000       4.210000       1.310000  1.731514e+06  
50%        19.670000      19.910000      11.460000  4.649648e+06  
75%        56.997500      57.570000      41.570000  1.102160e+07  
max       597.880000     605.400000     605.400000  9.276606e+09

,,remaining_issues
issue_type,severity,
extreme_return,medium,5057
missing_session,medium,5
ticker_normalization,low,36985
zero_volume,medium,74


,source_row_id,ts,ticker,open,high,low,close,adj_close,volume,ticker_original,ts_original,payload_signature,disposition,reason,repair_flags
99637,99637,1962-01-02,PG,0.0,1.44000,1.42000,1.43000,0.259300,192000.0,PG,1962-01-02,3875650174536127104,quarantined,unresolved_leading_price_gap,[]
224790,224790,1966-07-05,MCD,0.0,0.27366,0.26749,0.26955,0.118090,388800.0,MCD,1966-07-05,7918506836204265001,quarantined,unresolved_leading_price_gap,[]
287343,287343,1962-01-02,CVX,0.0,3.30000,3.24000,3.30000,0.335750,105840.0,CVX,1962-01-02,13306343275817044979,quarantined,unresolved_leading_price_gap,[]
350808,350808,1962-01-02,JNJ,0.0,0.22338,0.22222,0.22338,0.064012,0.0,JNJ,1962-01-02,12016949387724732379,quarantined,unresolved_leading_price_gap,[]
364568,364568,1962-01-02,MRK,0.0,0.39703,0.39317,0.39537,0.068447,633830.0,MRK,1962-01-02,3237797820759946639,quarantined,unresolved_leading_price_gap,[]


## 5. Required injected examples
These deliberately small **synthetic** cases demonstrate faults even if the actual database has none. They cover an intentionally removed session, grouped fills, unresolved leading prices, all-missing removal, a fill outside today's range, bounds repair, nonpositive/nonfinite prices, volume provenance, normalized duplicates, conflicts, split-date adjustment, undefined volatility, both return tails, and interior-only interpolation. Assertions make the demonstrated behavior checkable.

In [14]:
example_results = injected_examples()
display(example_results)
assert example_results.status.eq('PASS').all()

,case,input,retained,removed,quarantined,status
0,Missing cells: prior same-ticker price and dat...,6.0,6.0,0.0,0.0,PASS
1,Positive adjusted-price corruption: reconcile ...,6.0,6.0,0.0,0.0,PASS
2,All OHLC missing: remove before fill,6.0,5.0,1.0,0.0,PASS
3,Leading gap: never borrow another ticker,12.0,11.0,0.0,1.0,PASS
4,Forward-fill outside current range: quarantine,6.0,5.0,0.0,1.0,PASS
5,Impossible close with corroborated OHL: midpoi...,6.0,6.0,0.0,0.0,PASS
6,Nonpositive/nonfinite prices and volume,6.0,6.0,0.0,0.0,PASS
7,Normalize labels before exact duplicate removal,7.0,6.0,1.0,0.0,PASS
8,Conflicting keys: quarantine both candidates,7.0,5.0,0.0,2.0,PASS
9,One missing session: report without inventing ...,5.0,5.0,0.0,0.0,PASS


## 6. Persist the reproducible result
The cleaning notebook recreates the cleaned table each run, plus audit tables and CSV diagnostics. `ts` uses `YYYY-MM-DD` for inclusive SQL date filtering. Only the local working database is writable. The raw table must be unchanged after saving.

In [15]:
save_cleaned(result, db_path, audit_dir)
raw_after, _ = load_tables(db_path)
pd.testing.assert_frame_equal(raw, raw_after)
print("Saved ohlc_hw_cleaned and audit tables. Original ohlc_hw is unchanged.")
print("Audit directory:", Path(audit_dir))
print("Next: python 03_run_all_stocks.py")

Saved ohlc_hw_cleaned and audit tables. Original ohlc_hw is unchanged.
Audit directory: outputs/cleaning
Next: python 03_run_all_stocks.py
